# Explore Raw Data

Exploratory analysis of the bronze layer directly from PostgreSQL. This notebook examines both raw signal data and piece identification, then correlates them to understand the dataset before cleaning.

### What this notebook covers

1. **Raw signals** — record counts, signal types, value distributions, sampling frequency
2. **Combined view** — pieces with die matrix, cumulative travel times, and data quality
3. **Per die matrix** — travel time statistics, comparisons, variability
4. **Production patterns** — daily volumes, die matrix usage over time

### Column reference

All lifetime values are **cumulative piece travel times in seconds** from furnace exit.

| Signal | Process stage | Typical |
|---|---|---|
| `lifetime_first` | Main press — 2nd strike (1st op) | ~17–19s |
| `lifetime_second` | Main press — 3rd strike (2nd op) | ~24–26s |
| `lifetime_drill` | Main press — 4th strike (drill) | ~37–40s |
| `lifetime_auxiliary_press` | Auxiliary press | ~54–57s |
| `lifetime_bath` | Quench bath | ~56–59s |
| `lifetime` | General (≈ bath) | ~56–59s |
| `upsetting_lifetime` | Main press — 1st strike (upsetting) | ⚠️ Bad data, discard |

**Important**: These are per-piece travel times (~58s total), NOT OEE cycle time (11–16s between consecutive pieces).

In [2]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

env_path = Path("../infra/.env")
if not env_path.exists():
    raise FileNotFoundError(f"Could not find env file: {env_path.resolve()}")

env = {}
for line in env_path.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    k, v = line.split("=", 1)
    env[k.strip()] = v.strip()

host = "localhost"
port = env["POSTGRES_PORT"]
db = env["POSTGRES_DB"]
user = env["POSTGRES_USER"]
password = env["POSTGRES_PASSWORD"]

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
)

with engine.connect() as conn:
    print("Connected to PostgreSQL")
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())

raw_lifetime = pd.read_sql("SELECT * FROM bronze.raw_lifetime", engine)
raw_piece_info = pd.read_sql("SELECT * FROM bronze.raw_piece_info", engine)

print("raw_lifetime shape:", raw_lifetime.shape)
print("raw_piece_info shape:", raw_piece_info.shape)


Connected to PostgreSQL
Database: vaultech
raw_lifetime shape: (1233541, 3)
raw_piece_info shape: (359534, 3)


In [3]:
print(raw_lifetime.columns.tolist())
print(raw_piece_info.columns.tolist())

['timestamp', 'signal', 'value']
['timestamp', 'signal', 'value']


In [4]:
lifetime_signals = pd.read_sql("""
SELECT signal, COUNT(*) AS n
FROM bronze.raw_lifetime
GROUP BY signal
ORDER BY n DESC, signal
""", engine)

piece_info_signals = pd.read_sql("""
SELECT signal, COUNT(*) AS n
FROM bronze.raw_piece_info
GROUP BY signal
ORDER BY n DESC, signal
""", engine)

display(lifetime_signals)
display(piece_info_signals)

,signal,n
0,forging_line.auxiliary_press.maintenance.forgi...,184966
1,forging_line.general.maintenance.forging_line_...,179629
2,forging_line.bath.maintenance.forging_line_lif...,179628
3,forging_line.main_press.maintenance.forging_li...,179628
4,forging_line.main_press.maintenance.forging_li...,179628
5,forging_line.main_press.maintenance.forging_li...,179628
6,forging_line.main_press.maintenance.forging_li...,150434


,signal,n
0,forging_line.general.general.forging_line_die_...,179767
1,forging_line.general.general.forging_line_piec...,179767


In [5]:
signal_map = {
    "forging_line.main_press.maintenance.forging_line_lifetime_first_piecedata": "lifetime_2nd_strike",
    "forging_line.main_press.maintenance.forging_line_lifetime_second_piecedata": "lifetime_3rd_strike",
    "forging_line.main_press.maintenance.forging_line_lifetime_drill_piecedata": "lifetime_4th_strike",
    "forging_line.auxiliary_press.maintenance.forging_line_lifetime_auxiliary_press_piecedata": "lifetime_aux",
    "forging_line.bath.maintenance.forging_line_lifetime_bath_piecedata": "lifetime_bath",
    "forging_line.general.maintenance.forging_line_lifetime_piecedata": "lifetime_general",
    "forging_line.main_press.maintenance.forging_line_upsetting_lifetime_piecedata": "lifetime_upsetting",  # BAD
}
raw_lifetime["signal_name"] = raw_lifetime["signal"].map(signal_map)
raw_lifetime.head()

,timestamp,signal,value,signal_name
0,2025-11-06 15:25:02.416000+00:00,forging_line.bath.maintenance.forging_line_lif...,70.300003,lifetime_bath
1,2025-11-06 15:25:16.426000+00:00,forging_line.bath.maintenance.forging_line_lif...,56.200001,lifetime_bath
2,2025-11-06 15:25:29.134000+00:00,forging_line.bath.maintenance.forging_line_lif...,56.400002,lifetime_bath
3,2025-11-06 15:25:43.743000+00:00,forging_line.bath.maintenance.forging_line_lif...,56.900002,lifetime_bath
4,2025-11-06 15:25:56.462000+00:00,forging_line.bath.maintenance.forging_line_lif...,57.099998,lifetime_bath


---
## Part 1: Raw Signal Exploration

### Dataset overview

In [6]:
overview = pd.DataFrame({
    "table": ["bronze.raw_lifetime", "bronze.raw_piece_info"],
    "rows": [len(raw_lifetime), len(raw_piece_info)],
    "columns": [raw_lifetime.shape[1], raw_piece_info.shape[1]],
    "min_timestamp": [raw_lifetime["timestamp"].min(), raw_piece_info["timestamp"].min()],
    "max_timestamp": [raw_lifetime["timestamp"].max(), raw_piece_info["timestamp"].max()],
    "n_signals": [raw_lifetime["signal"].nunique(), raw_piece_info["signal"].nunique()],
})

display(overview)


,table,rows,columns,min_timestamp,max_timestamp,n_signals
0,bronze.raw_lifetime,1233541,4,2025-11-06 15:25:02.416000+00:00,2026-03-12 13:32:57.931000+00:00,7
1,bronze.raw_piece_info,359534,3,2025-11-06 15:25:02.416000+00:00,2026-03-11 09:50:50.453000+00:00,2


### Sample data

In [7]:
sample_ts = raw_lifetime["timestamp"].dropna().sort_values().iloc[0]

display(
    raw_lifetime.loc[raw_lifetime["timestamp"] == sample_ts]
    .sort_values("signal")
)


,timestamp,signal,value,signal_name
1048575,2025-11-06 15:25:02.416000+00:00,forging_line.auxiliary_press.maintenance.forgi...,68.699997,lifetime_aux
0,2025-11-06 15:25:02.416000+00:00,forging_line.bath.maintenance.forging_line_lif...,70.300003,lifetime_bath
179628,2025-11-06 15:25:02.416000+00:00,forging_line.general.maintenance.forging_line_...,70.300003,lifetime_general
898141,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,52.099998,lifetime_4th_strike
538885,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,32.000000,lifetime_2nd_strike
718513,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,38.700001,lifetime_3rd_strike
359257,2025-11-06 15:25:02.416000+00:00,forging_line.main_press.maintenance.forging_li...,0.200000,lifetime_upsetting


### Records per signal

In [8]:
records_per_signal = (
    raw_lifetime.groupby(["signal_name", "signal"], dropna=False)
    .size()
    .reset_index(name="n_records")
    .sort_values("n_records", ascending=False)
    .reset_index(drop=True)
)

display(records_per_signal)


,signal_name,signal,n_records
0,lifetime_aux,forging_line.auxiliary_press.maintenance.forgi...,184966
1,lifetime_general,forging_line.general.maintenance.forging_line_...,179629
2,lifetime_2nd_strike,forging_line.main_press.maintenance.forging_li...,179628
3,lifetime_bath,forging_line.bath.maintenance.forging_line_lif...,179628
4,lifetime_3rd_strike,forging_line.main_press.maintenance.forging_li...,179628
5,lifetime_upsetting,forging_line.main_press.maintenance.forging_li...,179628
6,lifetime_4th_strike,forging_line.main_press.maintenance.forging_li...,150434


### Value statistics per signal

All values are cumulative piece travel times in seconds from furnace exit.

In [9]:
value_stats = (
    raw_lifetime.groupby("signal_name")["value"]
    .agg(
        n="count",
        nulls=lambda s: s.isna().sum(),
        min="min",
        p01=lambda s: s.quantile(0.01),
        p05=lambda s: s.quantile(0.05),
        median="median",
        mean="mean",
        p95=lambda s: s.quantile(0.95),
        p99=lambda s: s.quantile(0.99),
        max="max",
        std="std",
    )
    .reset_index()
    .sort_values("median")
    .reset_index(drop=True)
)

display(value_stats)

,signal_name,n,nulls,min,p01,p05,median,mean,p95,p99,max,std
0,lifetime_upsetting,179628,0,0.0,0.0,0.000000,0.100000,0.095135,0.100000,0.700000,6.700000,0.160731
1,lifetime_2nd_strike,179628,0,0.0,0.0,16.000000,18.100000,20.392200,25.100000,82.145998,683.299988,16.045788
2,lifetime_3rd_strike,179628,0,0.0,0.0,22.799999,25.100000,27.318800,33.299999,89.172997,690.400024,16.291476
3,lifetime_4th_strike,150434,0,0.0,0.0,36.099998,38.500000,40.872465,50.599998,100.334000,716.799988,16.696892
4,lifetime_aux,184966,0,0.0,0.0,53.000000,56.700001,58.548445,69.000000,121.199997,734.900024,17.800796
5,lifetime_bath,179628,0,0.0,0.0,54.700001,58.400002,60.239980,70.800003,123.473000,736.599976,17.968613
6,lifetime_general,179629,0,0.0,0.0,54.700001,58.400002,60.264707,70.900002,123.671997,736.599976,17.949152


### Value distribution per signal

Percentile distribution and zero-value count. Zero values indicate tracking failures (~1.2% for most signals, 22.8% for upsetting — confirming it is bad data).

In [10]:
value_distribution = (
    raw_lifetime.groupby("signal_name")["value"]
    .agg(
        n="count",
        zero_count=lambda s: (s == 0).sum(),
        zero_pct=lambda s: (s == 0).mean() * 100,
        p00="min",
        p01=lambda s: s.quantile(0.01),
        p05=lambda s: s.quantile(0.05),
        p25=lambda s: s.quantile(0.25),
        p50="median",
        p75=lambda s: s.quantile(0.75),
        p95=lambda s: s.quantile(0.95),
        p99=lambda s: s.quantile(0.99),
        p100="max",
    )
    .reset_index()
    .sort_values("p50")
    .reset_index(drop=True)
)

display(value_distribution)


,signal_name,n,zero_count,zero_pct,p00,p01,p05,p25,p50,p75,p95,p99,p100
0,lifetime_upsetting,179628,40964,22.804908,0.0,0.0,0.000000,0.100000,0.100000,0.100000,0.100000,0.700000,6.700000
1,lifetime_2nd_strike,179628,2120,1.180217,0.0,0.0,16.000000,16.900000,18.100000,20.200001,25.100000,82.145998,683.299988
2,lifetime_3rd_strike,179628,2141,1.191908,0.0,0.0,22.799999,23.799999,25.100000,27.200001,33.299999,89.172997,690.400024
3,lifetime_4th_strike,150434,1866,1.240411,0.0,0.0,36.099998,37.200001,38.500000,40.900002,50.599998,100.334000,716.799988
4,lifetime_aux,184966,2225,1.202924,0.0,0.0,53.000000,54.700001,56.700001,58.900002,69.000000,121.199997,734.900024
5,lifetime_bath,179628,2171,1.208609,0.0,0.0,54.700001,56.400002,58.400002,60.599998,70.800003,123.473000,736.599976
6,lifetime_general,179629,2112,1.175757,0.0,0.0,54.700001,56.400002,58.400002,60.599998,70.900002,123.671997,736.599976


### Sampling frequency

Time interval between consecutive readings for the same signal. The median ~14s reflects the production cadence (one piece every ~14 seconds). Large max gaps (up to 353 hours) correspond to weekends or machine stops.

In [11]:
sampling_df = raw_lifetime[["timestamp", "signal_name"]].dropna().sort_values(["signal_name", "timestamp"]).copy()

sampling_df["delta_s"] = (
    sampling_df.groupby("signal_name")["timestamp"]
    .diff()
    .dt.total_seconds()
)

sampling_stats = (
    sampling_df.groupby("signal_name")["delta_s"]
    .agg(
        n_intervals="count",
        min_s="min",
        p05_s=lambda s: s.quantile(0.05),
        median_s="median",
        mean_s="mean",
        p95_s=lambda s: s.quantile(0.95),
        max_s="max",
    )
    .reset_index()
    .sort_values("median_s")
    .reset_index(drop=True)
)

display(sampling_stats)


,signal_name,n_intervals,min_s,p05_s,median_s,mean_s,p95_s,max_s
0,lifetime_4th_strike,150433,0.0,12.403,13.809,63.981574,26.0194,1272995.857
1,lifetime_aux,184965,0.0,12.407,13.911,58.820185,26.2150,1272995.857
2,lifetime_3rd_strike,179627,0.0,12.407,13.914,60.000920,26.3180,1272995.857
3,lifetime_2nd_strike,179627,0.0,12.407,13.914,60.000920,26.3180,1272995.857
4,lifetime_bath,179627,0.0,12.407,13.914,60.000920,26.3180,1272995.857
5,lifetime_general,179628,0.0,12.407,13.914,60.000586,26.3180,1272995.857
6,lifetime_upsetting,179627,0.0,12.407,13.914,60.000920,26.3180,1272995.857


---
## Part 2: Combined Piece View

Join lifetime signals with piece identification (piece_id, die_matrix) using the `bronze.v_pieces` view. This gives one row per piece with all cumulative times.

In [13]:
pieces = pd.read_sql("""
SELECT *
FROM bronze.v_pieces
ORDER BY timestamp
""", engine)

pieces["timestamp"] = pd.to_datetime(pieces["timestamp"], utc=True, errors="coerce")

print("shape:", pieces.shape)
print("columns:", pieces.columns.tolist())
display(pieces.head(10))

shape: (179765, 10)
columns: ['timestamp', 'piece_id', 'die_matrix', 'lifetime_1st_strike_s', 'lifetime_2nd_strike_s', 'lifetime_3rd_strike_s', 'lifetime_4th_strike_s', 'lifetime_auxiliary_press_s', 'lifetime_bath_s', 'lifetime_general_s']


,timestamp,piece_id,die_matrix,lifetime_1st_strike_s,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s
0,2025-11-06 15:25:02.416000+00:00,251106001720,5052,0.2,32.000000,38.700001,52.099998,68.699997,70.300003,70.300003
1,2025-11-06 15:25:16.426000+00:00,251106001721,5052,0.1,17.900000,24.600000,38.000000,54.599998,56.200001,56.200001
2,2025-11-06 15:25:29.134000+00:00,251106001722,5052,0.1,17.900000,24.600000,37.900002,54.799999,56.400002,56.400002
3,2025-11-06 15:25:43.743000+00:00,251106001723,5052,0.1,18.200001,24.799999,38.299999,55.299999,56.900002,56.900002
4,2025-11-06 15:25:56.462000+00:00,251106001724,5052,0.1,18.400000,25.100000,38.400002,55.500000,57.099998,57.099998
5,2025-11-06 15:26:10.462000+00:00,251106001725,5052,0.1,18.200001,24.799999,38.200001,55.500000,57.200001,57.200001
6,2025-11-06 15:26:23.771000+00:00,251106001726,5052,0.1,16.700001,23.400000,36.799999,53.599998,55.200001,55.200001
7,2025-11-06 15:26:36.382000+00:00,251106001727,5052,0.1,18.299999,24.900000,38.200001,54.700001,56.400002,56.400002
8,2025-11-06 15:26:50.095000+00:00,251106001728,5052,0.1,17.900000,24.500000,37.700001,54.700001,56.299999,56.299999
9,2025-11-06 15:27:17.711000+00:00,251106001729,5052,0.1,16.799999,23.500000,51.099998,68.099998,69.699997,69.699997


### Records per die matrix

How many records and unique pieces each die matrix has, and the time period it was active. Most production days show a single active matrix, but changeovers can happen mid-day.

In [14]:
records_per_matrix = (
    pieces.groupby("die_matrix")
    .agg(
        n_records=("piece_id", "size"),
        n_unique_pieces=("piece_id", "nunique"),
        first_timestamp=("timestamp", "min"),
        last_timestamp=("timestamp", "max"),
    )
    .reset_index()
)

records_per_matrix["active_days"] = (
    records_per_matrix["last_timestamp"] - records_per_matrix["first_timestamp"]
).dt.days + 1

records_per_matrix = records_per_matrix.sort_values("n_records", ascending=False).reset_index(drop=True)

display(records_per_matrix)


,die_matrix,n_records,n_unique_pieces,first_timestamp,last_timestamp,active_days
0,5090,87130,85549,2025-12-04 19:33:04+00:00,2026-02-17 22:38:20.199000+00:00,76
1,5091,53107,52392,2025-11-25 18:34:10.788000+00:00,2026-03-11 09:50:50.453000+00:00,106
2,5052,22843,22402,2025-11-06 15:25:02.416000+00:00,2026-02-25 11:32:58.186000+00:00,111
3,4974,16685,16428,2025-11-13 18:44:21.087000+00:00,2025-11-25 09:25:29.704000+00:00,12


---
## Part 3: Per Die Matrix Analysis

### Piece travel time statistics per die matrix

Each value is the elapsed time in seconds of a single piece traveling from furnace exit to a given process stage. These are NOT production cycle times (11–16s).

In [15]:
time_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
    "lifetime_general_s",
]

piece_travel_stats = (
    pieces.groupby("die_matrix")[time_cols]
    .agg(["count", "median", "mean", "std", "min", "max"])
    .round(2)
)

display(piece_travel_stats)


lifetime_2nd_strike_s                                   \
                           count median   mean    std  min    max   
die_matrix                                                          
4974                       16685   17.4  18.81  12.69  0.0  522.2   
5052                       22843   18.3  20.74  18.30  0.0  683.3   
5090                       87130   17.8  20.15  15.40  0.0  523.3   
5091                       52969   18.6  21.15  16.92  0.0  509.2   

           lifetime_3rd_strike_s                       ... lifetime_bath_s  \
                           count median   mean    std  ...            mean   
die_matrix                                             ...                   
4974                       16685   23.9  25.32  12.87  ...           57.03   
5052                       22843   25.4  27.68  18.53  ...           60.50   
5090                       87130   24.7  27.09  15.64  ...           60.14   
5091                       52969   25.7  28.16  17.19  ...           61.30   

                              lifetime_general_s                            \
              std  min    max              count median   mean    std  min   
die_matrix                                                                   
4974        14.47  0.0  560.2              16685   56.0  57.05  14.31  0.0   
5052        20.39  0.0  736.6              22843   58.5  60.38  20.56  0.0   
5090        17.30  0.0  568.7              87130   58.2  60.22  17.21  0.0   
5091        18.79  0.0  553.8              52970   59.2  61.29  18.82  0.0   

                   
              max  
die_matrix         
4974        560.2  
5052        736.6  
5090        568.7  
5091        553.8  

[4 rows x 36 columns]

### Travel time comparison across die matrices

Side-by-side median piece travel time (seconds) for each stage. Differences between matrices reflect die-specific tooling geometry and process parameters.

In [16]:
median_comparison = (
    pieces.groupby("die_matrix")[time_cols]
    .median()
    .round(2)
    .reset_index()
)

display(median_comparison)


,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s,lifetime_general_s
0,4974,17.4,23.9,37.2,54.2,56.0,56.0
1,5052,18.3,25.4,39.5,56.9,58.5,58.5
2,5090,17.8,24.7,38.7,56.6,58.2,58.2
3,5091,18.6,25.7,38.3,57.6,59.2,59.2


### Cumulative travel time profile per die matrix

The process order: **Furnace → 2nd strike (~18s) → 3rd strike (~25s) → 4th strike (~38s) → Aux. press (~55s) → Quench bath (~58s)**

Values must be monotonically increasing. Non-increasing values indicate measurement errors.

In [17]:
profile_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
]

cumulative_profile = (
    pieces.groupby("die_matrix")[profile_cols]
    .median()
    .round(2)
    .reset_index()
)

display(cumulative_profile)


,die_matrix,lifetime_2nd_strike_s,lifetime_3rd_strike_s,lifetime_4th_strike_s,lifetime_auxiliary_press_s,lifetime_bath_s
0,4974,17.4,23.9,37.2,54.2,56.0
1,5052,18.3,25.4,39.5,56.9,58.5
2,5090,17.8,24.7,38.7,56.6,58.2
3,5091,18.6,25.7,38.3,57.6,59.2


### Time spent between stages (per die matrix)

Computed by subtracting consecutive cumulative times. These partial times identify which segment causes delays.

| Partial | Calculation | What happens |
|---|---|---|
| Furnace → 2nd strike | `lifetime_2nd_strike_s` | Robot pick, transfer, positioning at main press |
| 2nd strike → 3rd strike | `lifetime_3rd - lifetime_2nd` | Press retraction, repositioning |
| 3rd strike → 4th strike | `lifetime_4th - lifetime_3rd` | Transfer to drill station |
| 4th strike → Aux. press | `lifetime_aux - lifetime_4th` | Exit main press, transfer to auxiliary press, deburring and coining |
| Aux. press → Bath | `lifetime_bath - lifetime_aux` | Transport from auxiliary press to quench bath |

In [18]:
partials = pieces.copy()

partials["t_furnace_to_2nd"] = partials["lifetime_2nd_strike_s"]
partials["t_2nd_to_3rd"] = partials["lifetime_3rd_strike_s"] - partials["lifetime_2nd_strike_s"]
partials["t_3rd_to_4th"] = partials["lifetime_4th_strike_s"] - partials["lifetime_3rd_strike_s"]
partials["t_4th_to_aux"] = partials["lifetime_auxiliary_press_s"] - partials["lifetime_4th_strike_s"]
partials["t_aux_to_bath"] = partials["lifetime_bath_s"] - partials["lifetime_auxiliary_press_s"]

partial_cols = [
    "t_furnace_to_2nd",
    "t_2nd_to_3rd",
    "t_3rd_to_4th",
    "t_4th_to_aux",
    "t_aux_to_bath",
]

partial_stats = (
    partials.groupby("die_matrix")[partial_cols]
    .agg(["count", "median", "mean", "std", "min", "max"])
    .round(2)
)

display(partial_stats)


t_furnace_to_2nd                                  t_2nd_to_3rd  \
                      count median   mean    std  min    max        count   
die_matrix                                                                  
4974                  16685   17.4  18.81  12.69  0.0  522.2        16685   
5052                  22843   18.3  20.74  18.30  0.0  683.3        22843   
5090                  87130   17.8  20.15  15.40  0.0  523.3        87130   
5091                  52969   18.6  21.15  16.92  0.0  509.2        52969   

                               ... t_4th_to_aux                     \
           median  mean   std  ...         mean   std   min    max   
die_matrix                     ...                                   
4974          6.5  6.51  1.74  ...        16.87  2.73   0.0   56.4   
5052          6.9  6.94  2.77  ...        17.24  3.49 -40.0  287.8   
5090          6.8  6.95  2.59  ...        17.48  2.91 -38.1  248.3   
5091          7.0  7.02  2.08  ...        16.93  3.06   0.0  312.7   

           t_aux_to_bath                                  
                   count median  mean   std   min    max  
die_matrix                                                
4974               16685    1.8  1.74  0.34   0.0   33.1  
5052               22843    1.6  1.62  1.22 -56.0  154.7  
5090               87130    1.6  1.60  0.72 -59.3  184.3  
5091               52969    1.6  1.64  1.90   0.0  302.3  

[4 rows x 30 columns]

### Zero values and anomalies per die matrix

- **Zeros**: tracking failures (value = 0.00s). Should be removed during cleaning.
- **Outliers (3×IQR)**: extreme values from stuck pieces, unclosed cycles, or machine stops.

In [21]:
time_cols = [
    "lifetime_2nd_strike_s",
    "lifetime_3rd_strike_s",
    "lifetime_4th_strike_s",
    "lifetime_auxiliary_press_s",
    "lifetime_bath_s",
]
zero_stats = (
    pieces.groupby("die_matrix")[time_cols]
    .apply(lambda df: pd.Series({
        f"{col}_zero_count": (df[col] == 0).sum()
        for col in time_cols
    }))
    .reset_index()
)

zero_pct = (
    pieces.groupby("die_matrix")[time_cols]
    .apply(lambda df: pd.Series({
        f"{col}_zero_pct": (df[col] == 0).mean() * 100
        for col in time_cols
    }))
    .reset_index()
)

zero_summary = zero_stats.merge(zero_pct, on="die_matrix")

display(zero_summary)
def iqr_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 3 * iqr
    upper = q3 + 3 * iqr
    return (series < lower) | (series > upper)

outlier_stats = []

for matrix, group in pieces.groupby("die_matrix"):
    row = {"die_matrix": matrix}
    
    for col in time_cols:
        s = group[col].dropna()
        
        if len(s) == 0:
            row[f"{col}_outlier_count"] = 0
            row[f"{col}_outlier_pct"] = 0
            continue
        
        mask = iqr_outlier_mask(s)
        row[f"{col}_outlier_count"] = mask.sum()
        row[f"{col}_outlier_pct"] = mask.mean() * 100
    
    outlier_stats.append(row)

outlier_summary = pd.DataFrame(outlier_stats)

display(outlier_summary)

,die_matrix,lifetime_2nd_strike_s_zero_count,lifetime_3rd_strike_s_zero_count,lifetime_4th_strike_s_zero_count,lifetime_auxiliary_press_s_zero_count,lifetime_bath_s_zero_count,lifetime_2nd_strike_s_zero_pct,lifetime_3rd_strike_s_zero_pct,lifetime_4th_strike_s_zero_pct,lifetime_auxiliary_press_s_zero_pct,lifetime_bath_s_zero_pct
0,4974,210,213,220,220,220,1.258616,1.276596,1.318550,1.318550,1.318550
1,5052,381,353,330,331,332,1.667907,1.545331,1.444644,1.449022,1.453399
2,5090,955,1003,1055,1057,1059,1.096063,1.151153,1.210834,1.213130,1.215425
3,5091,574,572,261,561,560,1.080837,1.077071,0.491461,1.056358,1.054475


,die_matrix,lifetime_2nd_strike_s_outlier_count,lifetime_2nd_strike_s_outlier_pct,lifetime_3rd_strike_s_outlier_count,lifetime_3rd_strike_s_outlier_pct,lifetime_4th_strike_s_outlier_count,lifetime_4th_strike_s_outlier_pct,lifetime_auxiliary_press_s_outlier_count,lifetime_auxiliary_press_s_outlier_pct,lifetime_bath_s_outlier_count,lifetime_bath_s_outlier_pct
0,4974,636,3.811807,677,4.057537,780,4.674858,1052,6.305064,1029,6.167216
1,5052,1439,6.299523,1531,6.702272,1611,7.052489,1489,6.518408,1552,6.794204
2,5090,3828,4.393435,4156,4.769884,4577,5.253070,4442,5.098129,4426,5.079766
3,5091,2543,4.800921,2717,5.129415,1223,5.144059,2610,4.914606,2592,4.893428


---
## Part 4: Production Patterns

### Daily production per die matrix

Number of pieces processed per day. Shows production volume, die matrix usage over time, and days with low counts (partial shifts, changeovers, maintenance).

In [22]:
pieces_daily = pieces.copy()
pieces_daily["production_date"] = pieces_daily["timestamp"].dt.date

daily_production = (
    pieces_daily.groupby(["production_date", "die_matrix"])
    .agg(
        n_records=("piece_id", "size"),
        n_unique_pieces=("piece_id", "nunique"),
    )
    .reset_index()
    .sort_values(["production_date", "die_matrix"])
)

display(daily_production)


,production_date,die_matrix,n_records,n_unique_pieces
0,2025-11-06,5052,1355,1328
1,2025-11-07,5052,3080,3040
2,2025-11-09,5052,381,373
3,2025-11-10,5052,885,875
4,2025-11-11,5052,3229,3169
...,...,...,...,...
83,2026-03-06,5091,1835,1816
84,2026-03-08,5091,437,434
85,2026-03-09,5091,3071,3019
86,2026-03-10,5091,3197,3155


### Daily record count per signal

In [23]:
raw_lifetime_daily = raw_lifetime.copy()
raw_lifetime_daily["production_date"] = raw_lifetime_daily["timestamp"].dt.date

daily_signal_counts = (
    raw_lifetime_daily.groupby(["production_date", "signal_name"])
    .size()
    .reset_index(name="n_records")
    .sort_values(["production_date", "signal_name"])
)

display(daily_signal_counts)


,production_date,signal_name,n_records
0,2025-11-06,lifetime_2nd_strike,1355
1,2025-11-06,lifetime_3rd_strike,1355
2,2025-11-06,lifetime_4th_strike,1355
3,2025-11-06,lifetime_aux,1355
4,2025-11-06,lifetime_bath,1355
...,...,...,...
573,2026-03-11,lifetime_aux,4352
574,2026-03-11,lifetime_bath,1154
575,2026-03-11,lifetime_general,1154
576,2026-03-11,lifetime_upsetting,1154
